# Session 04 · Part 4 — One agent, many users

Alice and Bob use the same agent. Every path looks the same to the agent, but the backend resolves it to data of the
current user. The user comes from the **runtime context** (`context=UserContext(...)`), which the application sets
from the authenticated request; the model never chooses it.

| Path | Store namespace | Who can write |
|---|---|---|
| `/memories/` | `("users", user_id, "memories")` | the user's agent |
| `/skills/approved/` | `("org", org_id, "skills")` | the application only (write denied to the agent) |
| `/skills/editable/` | `("users", user_id, "skills")` | the user's agent |
| `/workspace/` | `("users", user_id, "workspace")` | the user's agent |
| anything else | thread state (offloads, conversation history) | that thread |

<img src="images/b4_per_user_routes.png" alt="routes per user" width="640">

Docs: [Memory: user-scoped](https://docs.langchain.com/oss/python/deepagents/memory#user-scoped-memory) ·
[Skills: isolate skill libraries](https://docs.langchain.com/oss/python/deepagents/skills#isolate-skill-libraries) ·
[Permissions](https://docs.langchain.com/oss/python/deepagents/permissions)

## 0. Setup

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

OUT = Path("output")  # everything the agent writes goes here (git-ignored)
OUT.mkdir(exist_ok=True)

INFO | Model: gpt-6-luna


In [2]:
from utils import SKILLS_DIR, ask, context_window, internet_search, routes, system_prompt  # notebooks/utils.py

WINDOW = context_window(model)  # max input tokens, from the model's LangChain profile
logger.info("Context window: %s tokens", f"{WINDOW:,}")

INFO | Context window: 1,050,000 tokens


## 1. Runtime context, store and the shared skill library
The application seeds the approved skills of organization `acme` from `skills/` on disk. Agents can read them, not change them.

In [3]:
import re
from dataclasses import dataclass

from deepagents.backends.utils import create_file_data
from langgraph.store.memory import InMemoryStore


@dataclass
class UserContext:
    user_id: str  # in production: the verified identity of the request
    org_id: str


ORG = "acme"
APPROVED_NS = ("org", ORG, "skills")
store = InMemoryStore()
for path in sorted(SKILLS_DIR.rglob("*")):
    if path.is_file():
        key = "/" + path.relative_to(SKILLS_DIR).as_posix()
        store.put(APPROVED_NS, key, create_file_data(path.read_text()))
print(f"approved skills of {ORG}:", [i.key for i in store.search(APPROVED_NS)])

approved skills of acme: ['/research-brief/SKILL.md', '/research-brief/references/brief_template.md']


## 2. Routes that depend on the user
Each `namespace` is a function of the runtime: `rt.context.user_id` here (on LangGraph Server:
`rt.server_info.user.identity`). `FilesystemPermission` denies writes under `/skills/approved/`.
When two sources contain a skill with the same name, **the source listed last wins**, so the approved library is listed
last: a user's copy named `research-brief` in `/skills/editable/` cannot replace the shared one.

In [4]:
from deepagents import FilesystemPermission, create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import InMemorySaver


def user_ns(kind):
    return lambda rt: ("users", rt.context.user_id, kind)


# path the agent sees -> (store namespace of the current user or org, what the agent keeps there)
LAYOUT = {
    "/memories/": (StoreBackend(namespace=user_ns("memories")),
                   "AGENTS.md holds this user's lasting preferences; update it when they state one"),
    "/skills/approved/": (StoreBackend(namespace=lambda rt: ("org", rt.context.org_id, "skills")), None),  # read-only
    "/skills/editable/": (StoreBackend(namespace=user_ns("skills")),
                          "this user's own skills, one folder per skill with a SKILL.md"),
    "/workspace/": (StoreBackend(namespace=user_ns("workspace")), "this user's deliverables"),
}
backend = CompositeBackend(default=StateBackend(), routes=routes(LAYOUT))

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=system_prompt(LAYOUT),
    memory=["/memories/AGENTS.md"],
    skills=["/skills/editable/", "/skills/approved/"],  # last source wins on a name clash: approved last
    permissions=[FilesystemPermission(operations=["write"], paths=["/skills/approved/**"], mode="deny")],
    backend=backend,
    store=store,
    context_schema=UserContext,
    checkpointer=InMemorySaver(),
)


def chat(user_id: str, thread_id: str, question: str):
    """Stream one turn of this user; the thread id carries the user id."""
    print(f"[{user_id}]")
    return ask(agent, question, f"{user_id}-{thread_id}", window=WINDOW, context=UserContext(user_id=user_id, org_id=ORG))


def skills_in_prompt(log) -> dict:
    """Skill name -> SKILL.md path, read from the skills index part of the last system prompt."""
    index = log.system_parts(log.system)["skills index"]
    return dict(re.findall(r"- [*][*](.+?)[*][*]:.*?-> Read `(.+?)`", index, flags=re.S))

## 3. Alice: a preference and a personal skill
Thread ids carry the user id, so the threads of two users never collide.

In [5]:
result, log = chat("alice", "1",
    "Remember: always give me costs in euros and use metric units. Also create a personal skill for me at "
    "/skills/editable/term-glossary/SKILL.md: YAML frontmatter with name term-glossary and the description 'Build a short "
    "glossary of technical terms from a text. Use when the user asks for a glossary or a term list.', then three steps.")
print("\nalice in the store:", [(i.namespace[-1], i.key) for i in store.search(("users", "alice"))])

[alice]
→ ls({"path": "/memories"})
← ls success: No files found
→ ls({"path": "/skills/editable"})
← ls success: No files found
→ write_file({"file_path": "/skills/editable/term-glossary/SKILL.md", "content": "---\nname: term-glossary\ndescription: 'B…)
→ write_file({"file_path": "/memories/AGENTS.md", "content": "- Always give costs in euros and use metric units.\n"})
← write_file success: Updated file /skills/editable/term-glossary/SKILL.md
← write_file success: Updated file /memories/AGENTS.md
Saved your preference to always use euros for costs and metric units. I also created `/skills/editable/term-glossary/SKILL.md` with the requested YAML frontmatter and three steps.
call messages   tokens  in this call
   1        1    3,889  skills index, memory
   2        3    3,923  skills index, memory
   3        5    3,958  skills index, memory
   4        8    4,274  skills index, memory

alice in the store: [('memories', '/AGENTS.md'), ('skills', '/term-glossary/SKILL.md')]


Skills and memory are loaded once per thread, so Alice's new skill appears in her **next** thread. Her system prompt
now lists the approved skill and her own; the memory block holds her preference.

In [6]:
result, alice_log = chat("alice", "2", "Write a brief on heat pumps in cold climates.")
print("\nskills in Alice's prompt:", skills_in_prompt(alice_log))
print("alice workspace:", [i.key for i in store.search(("users", "alice", "workspace"))])

[alice]
→ read_file({"file_path": "/skills/approved/research-brief/SKILL.md", "offset": 0, "limit": 1000})
← read_file success: @@ lines 1-15 of 15 @@
---
name: research-brief
description: Write a one-page research brief with cited web so…
→ read_file({"file_path": "/skills/approved/research-brief/references/brief_template.md", "offset": 0, "limit": 1000})
→ internet_search({"query": "site:energy.gov cold climate heat pumps performance low temperatures field study", "max_results": 4…)
→ internet_search({"query": "site:nrel.gov cold climate heat pump field performance cold weather", "max_results": 4})
→ internet_search({"query": "IEA heat pumps cold climates efficiency cold temperature official report", "max_results": 4})
← read_file success: @@ lines 1-15 of 15 @@
# <Topic>: research brief

**Audience:** <who will read it>

## Question
One sentence: …


INFO | internet_search('IEA heat pumps cold climates efficiency cold temperature official rep…) -> ok 1748 ms


← internet_search success: [{"title": "Heat Pumps in Cold Climates: Winter Performance Guide 2026", "url": "https://a1solarstore.com/blog…


INFO | internet_search('site:energy.gov cold climate heat pumps performance low temperatures …) -> ok 3231 ms


← internet_search success: [{"title": "Field Performance of Inverter-Driven Heat Pumps in Cold ...", "url": "https://www1.eere.energy.gov…


INFO | internet_search('site:nrel.gov cold climate heat pump field performance cold weather') -> ok 5035 ms


← internet_search success: [{"title": "Cold Climate Field Study of the Effect of Defrost Controls on the Integrated Performance of a Duct…
→ internet_search({"query": "site:energy.gov cold climate heat pump field study COP low temperature capacity backup heating", "m…)


INFO | internet_search('site:energy.gov cold climate heat pump field study COP low temperatur…) -> ok 3184 ms


← internet_search success: [{"title": "Field Performance of Inverter-Driven Heat Pumps in Cold ...", "url": "https://www.energy.gov/sites…
→ write_file({"file_path": "/workspace/brief.md", "content": "# Heat pumps in cold climates: research brief\n\n**Audience:*…)
← write_file success: Updated file /workspace/brief.md
Cold-climate heat pumps can provide useful heat below −18 °C, but field-measured monthly COP varied from 1.1 to 3.1. [https://www.energy.gov/sites/prod/files/2015/09/f26/ba-case-study-inverter-driven-heat-pumps-cold.pdf]  
The brief covers performance evidence, capacity considerations, and remaining questions for city programs. [https://www.energy.gov/cmei/buildings/articles/split-system-cold-climate-heat-pump]  
Saved to `/workspace/brief.md`.
call messages   tokens  in this call
   1        1    3,891  skills index, memory
   2        3    4,172  skills index, memory
   3        8    6,133  skills index, memory
   4       10    6,919  skills index, memory
   5       12 

## 4. Bob: same agent, nothing of Alice's

In [7]:
result, bob_log = chat("bob", "1", "Which skills do you have, what do you remember about me, and what is in /workspace/?")
print("\nskills in Bob's prompt:", skills_in_prompt(bob_log), "\n")
alice_log.show(0)
print()
bob_log.show(0)  # same agent, a smaller system prompt: no personal skill, an empty memory file

[bob]
→ ls({"path": "/workspace/"})
→ ls({"path": "/skills/approved/"})
→ ls({"path": "/skills/editable/"})
→ ls({"path": "/memories/"})
← ls success: No files found
← ls success: ['/skills/approved/research-brief/']
← ls success: No files found
← ls success: No files found
- **Skill:** `research-brief` — for one-page, cited research briefs on technical or market topics.
- **What I remember about you:** Nothing is saved in memory.
- **`/workspace/`:** It’s empty.
call messages   tokens  in this call
   1        1    3,833  skills index, memory
   2        6    3,980  skills index, memory

skills in Bob's prompt: {'research-brief': '/skills/approved/research-brief/SKILL.md'} 

Call 1/5: 3,891 of 1,050,000 tokens (0.37%, provider count)
  [███████████████████▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓]  █ system prompt  ▓ tool schemas  ▒ messages
  System prompt                       1,833
    your system_prompt                  196
    skills index                        535
    memory: AGENTS.md content    

## 5. Try to break the isolation
Bob asks the agent to change the approved skill. The prompt's `<files>` section does not offer `/skills/approved/` for
writing, and if the agent tries anyway, the permission rule refuses the write before it reaches the store.
The agent may then save a modified copy under `/skills/editable/`; the cell below writes such a copy on purpose. Because
`/skills/approved/` is listed last, Bob's next thread still loads the approved version.

In [8]:
result, _ = chat("bob", "2", "Edit /skills/approved/research-brief/SKILL.md and delete the rule about citing urls.")
denied = [m.text for m in result["messages"] if m.type == "tool" and "permission denied" in m.text]
print("\nwrite attempted and denied by the permission rule:", bool(denied), "(the agent may also refuse first: <files> does not list /skills/approved/ as writable)")
# Make the bypass explicit: a modified copy with the same name in Bob's personal library (what the agent may write)
approved_text = store.get(APPROVED_NS, "/research-brief/SKILL.md").value["content"]
copy = approved_text.replace("- Cite only urls that appeared in a search result.\n", "")
print("the copy drops the citation rule:", copy != approved_text)
store.put(("users", "bob", "skills"), "/research-brief/SKILL.md", create_file_data(copy))
print("bob's personal skills now:", [i.key for i in store.search(("users", "bob", "skills"))])

result, bob_log2 = chat("bob", "3", "Which skills do you have? One line.")
print("research-brief is read from:", skills_in_prompt(bob_log2)["research-brief"])

[bob]
I can’t edit `/skills/approved/` because it’s outside the permitted write locations. I can create an editable copy under `/skills/editable/` with that URL-citation rule removed if you’d like.
call messages   tokens  in this call
   1        1    3,834  skills index, memory

write attempted and denied by the permission rule: False (the agent may also refuse first: <files> does not list /skills/approved/ as writable)
the copy drops the citation rule: True
bob's personal skills now: ['/research-brief/SKILL.md']
[bob]
research-brief — creates cited one-page research briefs on technical or market topics.
call messages   tokens  in this call
   1        1    3,822  skills index, memory
research-brief is read from: /skills/approved/research-brief/SKILL.md


## 6. Isolation report
Each check reads the store directly, not the agent's answer.

In [9]:
approved = store.get(APPROVED_NS, "/research-brief/SKILL.md").value["content"]
system_tokens = lambda log: sum(log.sizes(0)["system"].values())
checks = {
    "Alice's memory saved under her namespace": store.get(("users", "alice", "memories"), "/AGENTS.md") is not None,
    "Bob has no memory file": store.get(("users", "bob", "memories"), "/AGENTS.md") is None,
    "Alice's personal skill is in her prompt only": "term-glossary" in skills_in_prompt(alice_log)
        and "term-glossary" not in skills_in_prompt(bob_log),
    "Bob's workspace is empty": not store.search(("users", "bob", "workspace")),
    "Approved skill unchanged": "Cite only urls" in approved,
    "Bob's next thread loads the approved research-brief": skills_in_prompt(bob_log2)["research-brief"].startswith("/skills/approved/"),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\nsystem prompt: Alice {system_tokens(alice_log):,} tokens, Bob {system_tokens(bob_log):,} tokens")
for user in ("alice", "bob"):
    print(f"{user} items in the store:", [(i.namespace[-1], i.key) for i in store.search(("users", user))])

PASS  Alice's memory saved under her namespace
PASS  Bob has no memory file
PASS  Alice's personal skill is in her prompt only
PASS  Bob's workspace is empty
PASS  Approved skill unchanged
PASS  Bob's next thread loads the approved research-brief

system prompt: Alice 1,833 tokens, Bob 1,765 tokens
alice items in the store: [('memories', '/AGENTS.md'), ('skills', '/term-glossary/SKILL.md'), ('workspace', '/brief.md')]
bob items in the store: [('skills', '/research-brief/SKILL.md')]


### Try it

In [10]:
question = "What do you remember about me?"  # try both users
# result, log = chat("bob", "9", question)

### Done when
- All six checks print `PASS`.
- Alice's prompt lists `research-brief` and `term-glossary`; Bob's lists only `research-brief`.
- In production, `user_id` comes from the verified token (Session 09), never from the model or the message.